# Q-Learning Offline Algorithm

Unlike the $\epsilon$-greedy Monte Carlo algorithm implemented in 3.1.2.c), Q-Learning is an offline TD(0) method. This means that:

- actions are selected for exploration according to an $\epsilon$-greedy policy based on the state-action value function $Q_{\pi}(s,a)$;

- the function $Q_{\pi}(s,a)$ is updated after every transition using TD(0), as follows:
$$
Q(s,a) \leftarrow Q(s,a) + \alpha \left[ R + \gamma \max_{a' \in \mathcal{A}(s')} \left( Q(s',a') - Q(s,a) \right) \right] ;
$$

- a greedy target policy is learned, which selects, in each state, the action that maximizes the estimated value $Q_{\pi}(s,a)$, without encouraging exploration;

- the policy used for exploration differs from the policy the algorithm seeks to learn (the target policy), which is why Q-Learning is called offline.

In [51]:
import numpy as np

In [52]:
def epsilon_greedy_action(Q, s, epsilon):
    """
    Selects an action according to an epsilon-greedy policy,
    for exploration and exploitation.

    Q[s, a] : estimated value of state-action pair (s, a)
    s : current state
    epsilon : exploration rate in [0, 1]

    output : selected action
    """

    n_actions = Q.shape[1]

    # Exploration: choose a random action
    if np.random.random() < epsilon:
        action = np.random.choice(n_actions)

    # Exploitation: choose randomly among the best actions
    else:
        best_actions = np.flatnonzero(Q[s] == np.max(Q[s]))
        action = np.random.choice(best_actions)

    return action

In [53]:
def Q_learning(R_1, P_act, gamma, alpha, T, N, epsilon, absorbing_state=None):
    """
    Q-Learning algorithm with an epsilon-greedy policy.
    
    R_1[s, a] : matrix of rewards weighted by probabilities
    P_act[a, s, s'] : transition probabilities from s to s' under action a
    gamma : discount factor
    alpha : learning rate
    T : maximum number of steps per episode
    N : number of episodes
    epsilon : exploration rate in [0, 1]
    absorbing_state : list of absorbing states

    output:
    q_value : state-action value function
    pi : final epsilon-greedy policy
    """

    if absorbing_state is None:
        absorbing_state = []

    n_states, n_actions = R_1.shape

    # non absorbing states
    non_absorbing = [s for s in range(n_states) if s not in absorbing_state]
    if len(non_absorbing) == 0:
        raise ValueError("No non-absorbing states.")

    # Check dimensions
    assert P_act.shape == (n_actions, n_states, n_states)

    # Initialize Q(s, a) = 0
    q_value = np.zeros((n_states, n_actions))

    # Q-Learning
    for _ in range(N):

        # Select an initial state uniformly at random
        s = np.random.choice(non_absorbing)

        # Generate one episode / Estimate Q
        for t in range(T):

            # Stop if the current state is absorbing
            if s in absorbing_state:
                break

            # Choose an action according to epsilon-greedy
            a = epsilon_greedy_action(q_value, s, epsilon)

            # Select next state under P_act
            s_new = np.random.choice(n_states, p=P_act[a, s])

            # Observe the reward
            r = R_1[s, a]

            # Q-Learning update
            if s_new in absorbing_state:
                target = r
            else:
                target = r + gamma * np.max(q_value[s_new])

            q_value[s, a] += alpha * (target - q_value[s, a])

            # Move to the next state
            s = s_new

    # Construct the final epsilon-greedy policy
    # new policy
    pi = np.zeros((n_states, n_actions))

    for s in non_absorbing:

        best_actions = np.flatnonzero(q_value[s] == np.max(q_value[s]))

        # Exploration probability
        pi[s, :] = epsilon / n_actions

        # Exploitation probability
        pi[s, best_actions] += ( (1 - epsilon) / len(best_actions))


    return q_value, pi


In [ ]:
# Transition matrices for each action
P_0et = np.array([
    [0.5, 0.5, 0, 0, 0, 0, 0],
    [0, 0.6, 0, 0, 0.4, 0, 0],
    [0, 0.4, 0.6, 0, 0, 0, 0],
    [0, 0, 0, 0.1, 0, 0.9, 0],
    [0, 0, 0, 0, 1, 0, 0],
    [0, 0, 0, 0, 0, 1, 0],
    [0, 0, 0, 0, 0, 0, 1]
])

P_1et = np.array([
    [0.5, 0, 0.5, 0, 0, 0, 0],
    [0.3, 0, 0.7, 0, 0, 0, 0],
    [0, 0, 0.5, 0.5, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 1],
    [0, 0, 0, 0, 1, 0, 0],
    [0, 0, 0, 0, 0, 1, 0],
    [0, 0, 0, 0, 0, 0, 1]
])

P_act_etudiant = np.stack(
    [P_0et, P_1et], axis=0
)


# Rewards weighted by probabilities
R_1_etudiant = np.array([
    [0.5, -0.5],
    [-3.4, -0.7],
    [-0.2, -5.5],
    [89, -1000],
    [0, 0],
    [0, 0],
    [0, 0]
])


# Absorbing states
absorbing_state_et = [4, 5, 6]


# Call Q-Learning
Q, pi_optimal = Q_learning(
    R_1_etudiant,
    P_act_etudiant,
    gamma=0.8,
    alpha=0.01,
    T=100,
    N=1000,
    epsilon=0.1,
    absorbing_state=absorbing_state_et
)

print("Fonction de valeur des etats et actions :\n", Q)
print("\nPolitique optimale :\n", pi_optimal)


Fonction de valeur des etats et actions :
 [[  20.74352571   10.13042517]
 [   1.33505654   34.34604029]
 [  14.30031646   56.8675325 ]
 [  98.96652922 -490.01425375]
 [   0.            0.        ]
 [   0.            0.        ]
 [   0.            0.        ]]

Politique optimale :
 [[0.95 0.05]
 [0.05 0.95]
 [0.05 0.95]
 [0.95 0.05]
 [0.   0.  ]
 [0.   0.  ]
 [0.   0.  ]]
